```mermaid
graph LR
 日本語+Pythonのコーパス作成 --> トークナイザーの学習
 トークナイザーの学習 --> GPTモデルの設計
  GPTモデルの設計 --> 学習ループの実装
  学習ループの実装 --> ファインチューニング
  ファインチューニング --> 推論
```

| Step | Name |Description |
| ---- | ----------- |----------- |
| 1 | 日本語+Pythonのコーパス作成 | (1)日本語(20~200MB) --> Wikipedia, 青空文庫, ニュース系コーパス, (2)Python(20~200MB) --> GitHub, Kaggle Notebooks |
|2 | トークナイザーの学習 | 語彙数(16k~32k), 日本語(BPE < Unigram), Python code(インデントや記号を分離しすぎない) |
|3| GPTモデルの設計 | Embedding(token embedding, position embedding), Transformer Block(Multi-Head Attention, MLP, Layer Normalization, Residual), Language Model Heads(Softmax) |
|4| 学習ループの実装 | batch size(16~48), sequence length(256~512), learning time(1h~1day) |
|5| ファインチューニング | Kaggle Notebooks, Python code, Python Q&A |
|6| 推論 ||

In [ ]:
!pip install sentencepiece datasets

In [ ]:
from datasets import load_dataset
import pandas as pd
import os
import spacy
import sentencepiece as spm
import torch
import torch.nn as nn
from torch.nn import functional as F
from datasets.utils.version import dataclass

In [ ]:
torch.cuda.is_available()

In [ ]:
# Japanese datasets
ds_wiki = load_dataset("mini97/filtered_japanese-wikipedia")
ds_aozora = load_dataset("globis-university/aozorabunko-clean")

# Python datasets
ds_python = load_dataset("Arjun-G-Ravi/Python-codes")

In [ ]:
print(type(ds_wiki))
print(type(ds_aozora))
print(type(ds_python))

In [ ]:
ds_wiki["train"][0]

In [ ]:
ds_aozora["train"][0]

In [ ]:
ds_python["train"][0]

In [ ]:
def build_corpus(path):
  def normalize(text: str) -> str:
      """自然言語の改行や余計な空白を整形"""
      return text.replace("\n", " ").strip()

  def flatten_code(code: str) -> str:
      """Pythonコードを1行にflatten（改行→<NL>）"""
      code = code.strip().replace("\r\n", "\n")
      code = code.replace("\n", " <NL> ")
      return code.strip()

  # --- STEP 1: ウィキをサンプリング ---
  ds_wiki_train = ds_wiki["train"].shuffle(seed=42).select(range(10000))

  datasets = [
      (ds_wiki_train, "original"),         # 自然文
      (ds_aozora["train"], "text"),        # 自然文
      (ds_python["train"], ["code","question"])  # コード＋質問
  ]

  BUFFER_SIZE = 1000


  # ---------------------------------------------------------
  # ① wiki & aozora（自然文） → 句点「。」で文分割
  # ---------------------------------------------------------
  buffer = []
  with open(path, "w", encoding="utf-8") as f:
      for ds, column in datasets[:2]:   # wiki & aozora
          for row in ds:
              text = row[column]
              if not text:
                  continue

              # 句点で文分割
              sentences = text.split("。")

              for s in sentences:
                  s = s.strip()
                  if not s:
                      continue
                  s = normalize(s + "。")  # 文末に「。」を戻す

                  buffer.append(s)
                  if len(buffer) >= BUFFER_SIZE:
                      f.write("\n".join(buffer) + "\n")
                      buffer = []

      if buffer:
          f.write("\n".join(buffer) + "\n")
          buffer = []


  # ---------------------------------------------------------
  # ② python（コード + 質問） → flatten_code + normalize
  # ---------------------------------------------------------
  buffer = []
  with open(path, "a", encoding="utf-8") as f:
      for ds, columns in datasets[2:3]:

          code_col, q_col = columns  # "code", "question"

          for row in ds:
              code_text = row[code_col]
              q_text = row[q_col]

              if not code_text and not q_text:
                  continue

              # 結合 → flatten → normalize
              combined = f"{code_text} {q_text}"
              combined = flatten_code(combined)
              combined = normalize(combined)

              buffer.append(combined)
              if len(buffer) >= BUFFER_SIZE:
                  f.write("\n".join(buffer) + "\n")
                  buffer = []

      if buffer:
          f.write("\n".join(buffer) + "\n")
          buffer = []


  print("✔ corpus.txt が完成しました！")
  return None

In [ ]:
os.makedirs('data', exist_ok=True)

if os.path.exists("data/corpus.txt"):
  print("corpus.txt exists -> skip")
else:
  print("Going to create corpus.txt")
  build_corpus("data/corpus.txt")
  print("Created corpus.txt")

In [ ]:
# SentencePiece を使って学習
# Tokenizer
def train_spm():
  spm.SentencePieceTrainer.train(
      input='data/corpus.txt',        # ← 学習に使うコーパス（1文1行が望ましい）
      model_prefix='model/spm_unigram_32k', # ← 出力ファイル名のプレフィックス
                                      #    spm_unigram_32k.model / spm_unigram_32k.vocab が生成される

      vocab_size=32000,               # ← 語彙数（サブワード数）。32k は LLM に最も一般的
                                      #    16k → 小型モデル向け、50k〜80k → 大型モデル向け

      model_type='unigram',           # ← subword のタイプ。日本語は “unigram” が最適
                                      #    BPE より文節ごとの自然さが出るため推奨

      character_coverage=0.9995,      # ← 収録する文字のカバー率
                                      #    日本語では 0.9995〜1.0 推奨
                                      #    稀な漢字もほぼすべて vocab に入るようにする

      user_defined_symbols=['<NL>'],  # ← 特殊トークンを定義
                                      #    flatten_code() の "<NL>" が1トークンとして扱われる
                                      #    追加例: ['<NL>', '<code>', '</code>']

      bos_id=-1,                      # ← BOS（文頭）トークンを無効化（-1 は無効の意味）
      eos_id=-1,                      # ← EOS（文末）トークンを無効化
      pad_id=-1,                      # ← PAD を無効化（pretrain コーパスでは不要）
      unk_id=0                        # ← UNK（未知語）トークンを id=0 に設定（必須）
  )


In [ ]:
# modelフォルダの作成（念のため）
os.makedirs('model', exist_ok=True)
print("✅ model フォルダの準備完了。")

if os.path.exists("model/spm_unigram_32k.model"):
  print("spm_unigram_32k.model exists -> skip")
else:
  print("Going to create spm_unigram_32k.model")
  train_spm()
  print("Created spm_unigram_32k.model")

In [ ]:
@dataclass
class MiniGPTConfig:
  vocab_size: int = 32000
  dim: int = 512
  n_layers: int = 6
  n_heads: int = 8
  ff_dim: int = 2048
  max_seq_len: int = 256
  rotary_pct: float = 1.0
  dropout: float = 0.2
  device: str = "cuda" if torch.cuda.is_available() else "cpu"

class MiniGPT(nn.Module):
  def __init__(self, config:MiniGPTConfig):
    super().__init__()
    self.config = config
    self.token_emb = token_emb
    self.pos_emb = pos_emb
    self.blocks = nn.ModuleList([Block(config) for _ in range(config.n_layers)])
    self.ln = nn.LayerNorm(config.dim)
    self.head = nn.Linear(config.dim, config.vocab_size)

  def forward(self, input_ids):


    return logits

